# Ingesta y verificación de datos — Entrega 1

Proyecto Integrador de Ciencia de Datos (UTN FRM 2026): **¿qué combinación de nivel de ELO, apertura jugada, modalidad de ritmo y color de piezas predice mejor quién gana una partida de ajedrez online, y qué tan larga es?**

**Unidad de análisis**: una partida de ajedrez individual jugada en Chess.com.

**Fuente**: [PubAPI pública de Chess.com](https://www.chess.com/news/view/published-data-api), sin autenticación. Para cada cuenta se consultan sus archivos mensuales desde el más reciente hacia atrás, con acceso estrictamente secuencial. Se descargan partidas reales de jugadores en distintos rangos de rating, desde nivel club hasta élite mundial.

Este notebook **no reimplementa la lógica del pipeline**: importa las clases de `src/` y las ejecuta acá, con narrativa explicando cada decisión. El pipeline también puede correrse fuera del notebook con `python -m src.pipeline`.

In [1]:
# Imports and base paths.
# The notebook's working directory is switched to the repo root so that the
# relative paths in config.yaml (data/raw, data/processed) resolve the same
# way whether the pipeline runs from the notebook or from `python -m src.pipeline`.
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path("..").resolve()
if Path.cwd() != REPO_ROOT:
    os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from src.clean_data import DataCleaner
from src.download_data import DataDownloader
from src.feature_engineering import FeatureEngineer
from src.pipeline import build_summary
from src.utils import load_config

CONFIG_PATH = Path("config/config.yaml")
config = load_config(CONFIG_PATH)

## Sección 1 — Descarga automatizada

`DataDownloader` consulta primero `/pub/player/{username}/games/archives` y luego los archivos mensuales `/games/{YYYY}/{MM}`. Recorre los meses en orden descendente hasta reunir como máximo 1.000 partidas rated de ajedrez estándar en bullet, blitz o rapid por usuario. Las respuestas se guardan como JSON y la descarga es idempotente: si el archivo ya existe en `data/raw/`, se saltea.

In [2]:
# Download monthly game archives for every configured user.
downloader = DataDownloader(config)
raw_paths = downloader.download_all()
raw_paths

2026-08-24 17:42:06 - src.download_data - INFO - Ya existe data/raw/chesscom_RebeccaHarris_raw.json — se saltea la descarga de RebeccaHarris.


2026-08-24 17:42:07 - src.download_data - INFO - Ya existe data/raw/chesscom_erik_raw.json — se saltea la descarga de erik.


2026-08-24 17:42:07 - src.download_data - INFO - Ya existe data/raw/chesscom_AnnaCramling_raw.json — se saltea la descarga de AnnaCramling.


2026-08-24 17:42:07 - src.download_data - INFO - Ya existe data/raw/chesscom_AlexandraBotez_raw.json — se saltea la descarga de AlexandraBotez.


2026-08-24 17:42:07 - src.download_data - INFO - Ya existe data/raw/chesscom_GothamChess_raw.json — se saltea la descarga de GothamChess.


2026-08-24 17:42:08 - src.download_data - INFO - Ya existe data/raw/chesscom_IMRosen_raw.json — se saltea la descarga de IMRosen.


2026-08-24 17:42:08 - src.download_data - INFO - Ya existe data/raw/chesscom_chessbrah_raw.json — se saltea la descarga de chessbrah.


2026-08-24 17:42:08 - src.download_data - INFO - Ya existe data/raw/chesscom_hikaru_raw.json — se saltea la descarga de hikaru.


{'RebeccaHarris': PosixPath('data/raw/chesscom_RebeccaHarris_raw.json'),
 'erik': PosixPath('data/raw/chesscom_erik_raw.json'),
 'AnnaCramling': PosixPath('data/raw/chesscom_AnnaCramling_raw.json'),
 'AlexandraBotez': PosixPath('data/raw/chesscom_AlexandraBotez_raw.json'),
 'GothamChess': PosixPath('data/raw/chesscom_GothamChess_raw.json'),
 'IMRosen': PosixPath('data/raw/chesscom_IMRosen_raw.json'),
 'chessbrah': PosixPath('data/raw/chesscom_chessbrah_raw.json'),
 'hikaru': PosixPath('data/raw/chesscom_hikaru_raw.json')}

## Sección 2 — Parseo y limpieza

Cada objeto JSON incluye metadata estructurada y una representación PGN completa. `DataCleaner` combina ambas: toma del JSON el identificador, la modalidad y la variante; del PGN extrae jugadores, ratings, resultado, ECO, terminación y jugadas. También deduplica por `GameUrl`, porque dos cuentas configuradas pueden aparecer en la misma partida. No hace falta `python-chess`: el proyecto analiza metadata y duración, no reconstruye posiciones.

In [3]:
# Parse, clean and validate all downloaded JSON archives.
cleaner = DataCleaner(config)
df, raw_row_count = cleaner.clean(raw_paths)
df = cleaner.optimize_dtypes(df)

print(f"Partidas crudas: {raw_row_count:,}")
print(f"Partidas válidas: {len(df):,} ({100 * len(df) / raw_row_count:.1f}% retenidas)")
df.head()

2026-08-24 17:42:08 - src.clean_data - INFO - Parseadas 215 partidas de RebeccaHarris


2026-08-24 17:42:09 - src.clean_data - INFO - Parseadas 1000 partidas de erik


2026-08-24 17:42:09 - src.clean_data - INFO - Parseadas 1000 partidas de AnnaCramling


2026-08-24 17:42:09 - src.clean_data - INFO - Parseadas 1000 partidas de AlexandraBotez


2026-08-24 17:42:09 - src.clean_data - INFO - Parseadas 1000 partidas de GothamChess


...

2026-08-24 17:42:09 - src.clean_data - INFO - Parseadas 1000 partidas de hikaru


2026-08-24 17:42:09 - src.clean_data - INFO - Se eliminaron 1 partidas repetidas entre usuarios.


2026-08-24 17:42:10 - src.clean_data - INFO - Limpieza completa: 7215 registros descargados -> 7208 partidas válidas (99.9%).


Partidas crudas: 7,215
Partidas válidas: 7,208 (99.9% retenidas)


,GameUrl,Event,Date,White,Black,Result,WhiteElo,BlackElo,Variant,TimeControl,TimeClass,ECO,Opening,Termination,Rated,moves_text,resultado,tiempo_base_seg,incremento_seg,cantidad_jugadas
0,https://www.chess.com/game/live/173348447610,Live Chess,2026-08-22,konowalow76,RebeccaHarris,1-0,1788,1750,Standard,600,rapid,C89,Ruy Lopez Opening Marshall Attack Modern Main ...,konowalow76 won by resignation,True,1. e4 1... e5 2. Nf3 2... Nc6 3. Bb5 3... a6 4...,Gana Blancas,600,0.0,130
1,https://www.chess.com/game/live/173324194654,Live Chess,2026-08-21,RebeccaHarris,welidm,1-0,1760,1723,Standard,600,rapid,A40,Englund Gambit 2.dxe5,RebeccaHarris won by resignation,True,1. d4 1... e5 2. dxe5 2... d6 3. exd6 3... Bxd...,Gana Blancas,600,0.0,129
2,https://www.chess.com/game/live/173324062816,Live Chess,2026-08-21,welidm,RebeccaHarris,0-1,1731,1750,Standard,600,rapid,C45,Scotch Game Classical Variation 5.Be3 Qf6,RebeccaHarris won by resignation,True,1. e4 1... e5 2. Nf3 2... Nc6 3. d4 3... exd4 ...,Gana Negras,600,0.0,14
3,https://www.chess.com/game/live/173122017970,Live Chess,2026-08-17,Nzoli1,RebeccaHarris,0-1,1711,1739,Standard,600,rapid,C00,French Defense,RebeccaHarris won by checkmate,True,1. e4 1... e6 2. Bc4 2... d5 3. exd5 3... exd5...,Gana Negras,600,0.0,62
4,https://www.chess.com/game/live/173121614602,Live Chess,2026-08-17,RebeccaHarris,Sajad_aftab484,1/2-1/2,1729,1732,Standard,600,rapid,B08,Pirc Defense Classical Variation 4,Game drawn by repetition,True,1. d4 1... Nf6 2. Nc3 2... g6 3. e4 3... d6 4....,Empate,600,0.0,61


## Sección 3 — Ingeniería de features

`FeatureEngineer.transform` agrega, todo vectorizado:

- `diferencia_elo`, `elo_promedio`, `favorito`: comparación del rating informado al cierre de la partida. Chess.com no ofrece en este endpoint un snapshot estrictamente prepartida; la limitación se documenta para el modelado posterior.
- `nivel_promedio`: banda de ELO de la partida (principiante a top mundial).
- `modalidad`: bullet / blitz / rapid según la clasificación `time_class` provista por Chess.com.
- `es_sorpresa`: 1 si ganó el jugador con menor ELO. **Target de clasificación adicional.**
- `familia_apertura`: agrupación de la apertura jugada por su letra ECO (A-E).

In [4]:
# Feature engineering
engineer = FeatureEngineer(config)
df = engineer.transform(df)
df.head()

2026-08-24 17:42:10 - src.feature_engineering - INFO - Generando features sobre 7208 partidas...


2026-08-24 17:42:10 - src.feature_engineering - INFO - Features generadas: ['GameUrl', 'Event', 'Date', 'White', 'Black', 'Result', 'WhiteElo', 'BlackElo', 'Variant', 'TimeControl', 'TimeClass', 'ECO', 'Opening', 'Termination', 'Rated', 'moves_text', 'resultado', 'tiempo_base_seg', 'incremento_seg', 'cantidad_jugadas', 'diferencia_elo', 'elo_promedio', 'favorito', 'nivel_promedio', 'modalidad', 'es_sorpresa', 'familia_apertura']


,GameUrl,Event,Date,White,Black,Result,WhiteElo,BlackElo,Variant,TimeControl,...,tiempo_base_seg,incremento_seg,cantidad_jugadas,diferencia_elo,elo_promedio,favorito,nivel_promedio,modalidad,es_sorpresa,familia_apertura
0,https://www.chess.com/game/live/173348447610,Live Chess,2026-08-22,konowalow76,RebeccaHarris,1-0,1788,1750,Standard,600,...,600,0.0,130,38,1769.0,Blancas,intermedio,Rapid,0,Abierta
1,https://www.chess.com/game/live/173324194654,Live Chess,2026-08-21,RebeccaHarris,welidm,1-0,1760,1723,Standard,600,...,600,0.0,129,37,1741.5,Blancas,intermedio,Rapid,0,Flanco
2,https://www.chess.com/game/live/173324062816,Live Chess,2026-08-21,welidm,RebeccaHarris,0-1,1731,1750,Standard,600,...,600,0.0,14,-19,1740.5,Negras,intermedio,Rapid,0,Abierta
3,https://www.chess.com/game/live/173122017970,Live Chess,2026-08-17,Nzoli1,RebeccaHarris,0-1,1711,1739,Standard,600,...,600,0.0,62,-28,1725.0,Negras,intermedio,Rapid,0,Abierta
4,https://www.chess.com/game/live/173121614602,Live Chess,2026-08-17,RebeccaHarris,Sajad_aftab484,1/2-1/2,1729,1732,Standard,600,...,600,0.0,61,-3,1730.5,Negras,intermedio,Rapid,0,Semiabierta


## Sección 4 — Exportación

Se guarda el dataset final en Parquet (compresión `snappy`), un sample CSV para inspección rápida y `data_summary.json` con métricas de volumen, targets y nulos.

In [5]:
# Export processed dataset
processed_dir = Path(config["paths"]["processed_dir"])
processed_dir.mkdir(parents=True, exist_ok=True)

clean_parquet_path = Path(config["paths"]["clean_parquet"])
df.to_parquet(clean_parquet_path, engine="pyarrow", compression="snappy", index=False)

sample_path = Path(config["paths"]["clean_sample_csv"])
sample_size = min(config["processing"]["sample_size"], len(df))
df.sample(n=sample_size, random_state=config["processing"]["random_state"]).to_csv(
    sample_path, index=False
)

summary = build_summary(df, raw_row_count)
summary_path = Path(config["paths"]["summary_json"])
with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False, default=str)

print(f"Parquet: {clean_parquet_path}")
print(f"Sample CSV ({sample_size} filas): {sample_path}")
print(f"Resumen JSON: {summary_path}")

Parquet: data/processed/partidas_ajedrez_clean.parquet
Sample CSV (5000 filas): data/processed/partidas_ajedrez_clean_sample.csv
Resumen JSON: data/processed/data_summary.json


## Sección 5 — Verificación

Se recarga el Parquet recién escrito (no el `df` en memoria) para confirmar que lo persistido es correcto de punta a punta. En la corrida de validación se descartaron siete registros: un duplicado porque dos cuentas configuradas jugaron entre sí y seis partidas rated con resultado pero sin movimientos (abortos o resoluciones administrativas). No se ocultan: el filtro exige al menos un ply porque la duración es uno de los targets. El dataset final no contiene nulos.

In [6]:
# Reload the persisted parquet to verify it independently of the in-memory df
df_check = pd.read_parquet(clean_parquet_path)
df_check.info()

<class 'pandas.DataFrame'>
RangeIndex: 7208 entries, 0 to 7207
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   GameUrl           7208 non-null   str           
 1   Event             7208 non-null   category      
 2   Date              7208 non-null   datetime64[us]
 3   White             7208 non-null   str           
 4   Black             7208 non-null   str           
 5   Result            7208 non-null   str           
 6   WhiteElo          7208 non-null   int16         
 7   BlackElo          7208 non-null   int16         
 8   Variant           7208 non-null   str           
 9   TimeControl       7208 non-null   str           
 10  TimeClass         7208 non-null   category      
 11  ECO               7208 non-null   category      
 12  Opening           7208 non-null   category      
 13  Termination       7208 non-null   category      
 14  Rated             7208 non-null   b

In [7]:
df_check.describe()

,Date,WhiteElo,BlackElo,tiempo_base_seg,incremento_seg,cantidad_jugadas,diferencia_elo,elo_promedio,es_sorpresa
count,7208,7208.000000,7208.000000,7208.0,7208.0,7208.000000,7208.000000,7208.000000,7208.000000
mean,2026-02-19 21:09:47.347391,2597.205189,2594.177858,132.984323,0.180522,85.900388,3.027331,2595.691523,0.236265
min,2023-11-12 00:00:00,732.000000,218.000000,10.0,0.0,1.000000,-1514.000000,865.500000,0.000000
25%,2026-01-19 00:00:00,2299.000000,2295.000000,60.0,0.0,59.000000,-97.000000,2306.375000,0.000000
50%,2026-07-08 12:00:00,2778.000000,2774.000000,60.0,0.0,80.000000,2.000000,2790.500000,0.000000
75%,2026-08-01 00:00:00,2993.000000,2993.250000,180.0,0.0,109.000000,104.000000,2996.000000,0.000000
max,2026-08-24 00:00:00,3468.000000,3469.000000,900.0,10.0,272.000000,2023.000000,3387.500000,1.000000
std,NaN,535.636692,537.766628,111.646112,0.494457,35.654768,220.471672,525.259809,0.424816


In [8]:
# Null check after filtering and deriving opening names from ECO URLs.
nulls = df_check.isna().sum()
nulls[nulls > 0]

Series([], dtype: int64)

In [9]:
# Target variable distributions
print("resultado:")
print(df_check["resultado"].value_counts(normalize=True))
print("\nmodalidad:")
print(df_check["modalidad"].value_counts())
print("\nnivel_promedio:")
print(df_check["nivel_promedio"].value_counts())
print(f"\nTasa de sorpresas (gana el de menor ELO): {100 * df_check['es_sorpresa'].mean():.1f}%")

resultado:
resultado
Gana Blancas    0.492647
Gana Negras     0.443674
Empate          0.063679
Name: proportion, dtype: float64

modalidad:
modalidad
Bullet    3887
Blitz     3099
Rapid      222
Name: count, dtype: int64

nivel_promedio:
nivel_promedio
top_mundial     4004
experto         1917
intermedio      1156
avanzado         121
principiante      10
Name: count, dtype: int64

Tasa de sorpresas (gana el de menor ELO): 23.6%


In [10]:
# Primer vistazo a la pregunta de investigación: probabilidad de sorpresa
# según qué tan grande es la diferencia de ELO entre los jugadores.
abs_diferencia_elo = df_check["diferencia_elo"].abs()
bins = pd.cut(abs_diferencia_elo, bins=[0, 50, 100, 200, 400, 5000])
tasa_sorpresa_por_brecha = df_check.groupby(bins, observed=True)["es_sorpresa"].mean()
print(tasa_sorpresa_por_brecha)

fig, ax = plt.subplots(figsize=(7, 4))
tasa_sorpresa_por_brecha.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("Tasa de sorpresa")
ax.set_xlabel("Diferencia de ELO absoluta")
ax.set_title("¿Cuánto importa la diferencia de ELO para que gane el 'underdog'?")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

diferencia_elo
(0, 50]        0.308344
(50, 100]      0.304888
(100, 200]     0.241681
(200, 400]     0.124821
(400, 5000]    0.038229
Name: es_sorpresa, dtype: float64


/tmp/ipykernel_162220/1795766995.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Move count by time class — duration sanity check.
fig, ax = plt.subplots(figsize=(7, 4))
df_check.boxplot(column="cantidad_jugadas", by="modalidad", ax=ax)
ax.set_ylabel("Cantidad de jugadas (plies)")
ax.set_title("Duración de la partida por modalidad de ritmo")
plt.suptitle("")
plt.tight_layout()
plt.show()

/tmp/ipykernel_162220/319120101.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Final integrity checks
assert df_check.shape[0] > 1_000, "El volumen final es insuficiente para la entrega"
assert df_check["GameUrl"].is_unique, "Quedaron partidas duplicadas"
assert df_check["resultado"].isin(["Gana Blancas", "Gana Negras", "Empate"]).all(), "resultado tiene valores inesperados"
assert df_check["cantidad_jugadas"].min() > 0, "Toda partida válida debe tener al menos 1 jugada"
assert df_check[["WhiteElo", "BlackElo"]].isna().sum().sum() == 0, "No debe haber nulos en ELO"
assert df_check["es_sorpresa"].isin([0, 1]).all(), "es_sorpresa debe ser binaria"
configured = {user.lower() for user in config["chess_com"]["usernames"]}
observed = set(df_check["White"].str.lower()) | set(df_check["Black"].str.lower())
assert configured <= observed, "Falta al menos un usuario configurado en el dataset"

print(f"Shape final: {df_check.shape}")
print("Todas las verificaciones pasaron correctamente.")

Shape final: (7208, 27)
Todas las verificaciones pasaron correctamente.
